# Checks

Run All after every change in `src/netad/`. Stops at the first failing assert.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score

from netad.data import (
    DATA_DIR, FEATURES, CATEGORICAL, NUMERIC, TARGETS, EXPECTED_COLUMNS,
    load_unsw_frame, load_unsw, deduplicate, resolve_conflicts, overlap_with_train, distinct_rows,
    train_val_split,
)
from netad.preprocessing import build_preprocessor, TTL_COLUMNS
from netad.evaluation import (
    evaluate_binary, choose_threshold, evaluate_multiclass, evaluate_test,
    detection_by_category, fpr_by_group,
    save_result, results_table, plot_roc_pr, plot_confusion,
)
from netad.models.autoencoder import (
    Autoencoder, reconstruction_error, train_autoencoder, save_autoencoder, load_autoencoder,
)

## data.py

### Schema

In [3]:
assert DATA_DIR.is_dir(), DATA_DIR
assert len(CATEGORICAL) == 3
assert len(NUMERIC) == 39
assert len(FEATURES) == 42
assert set(CATEGORICAL).isdisjoint(NUMERIC)
assert len(set(EXPECTED_COLUMNS)) == len(EXPECTED_COLUMNS) == 45

### load_unsw_frame

In [4]:
train_df = load_unsw_frame("train")
test_df = load_unsw_frame("test")

assert train_df.shape == (175341, 45), train_df.shape
assert test_df.shape == (82332, 45), test_df.shape
assert train_df.columns[0] == "id", train_df.columns[0]
assert set(train_df.columns) == set(test_df.columns) == set(EXPECTED_COLUMNS)
assert train_df.isna().sum().sum() == 0
assert test_df.isna().sum().sum() == 0

### load_unsw

In [5]:
X_train, y_train, y_train_multi = load_unsw("train")
X_test, y_test, y_test_multi = load_unsw("test")

assert X_train.shape == (175341, 42), X_train.shape
assert X_test.shape == (82332, 42), X_test.shape
assert list(X_train.columns) == FEATURES
assert not {"id", "label", "attack_cat"} & set(X_train.columns)
assert set(y_train.unique()) == set(y_test.unique()) == {0, 1}
assert ((y_train_multi == "Normal") == (y_train == 0)).all()
assert ((y_test_multi == "Normal") == (y_test == 0)).all()
assert y_train_multi.nunique() == y_test_multi.nunique() == 10
assert all(pd.api.types.is_string_dtype(X_train[c]) for c in CATEGORICAL)
assert all(pd.api.types.is_numeric_dtype(X_train[c]) for c in NUMERIC)

### deduplicate

In [6]:
dedup = deduplicate(train_df)

assert len(dedup) == 107740, len(dedup)
assert not dedup.duplicated(FEATURES + TARGETS).any()
assert dedup.index.isin(train_df.index).all()
assert len(train_df) == 175341

### resolve_conflicts

Row counts after `resolve_conflicts` + `deduplicate`.

In [7]:
expected = {
    ("label", "drop"): 107282,
    ("label", "majority"): 107412,
    ("attack_cat", "drop"): 99268,
    ("attack_cat", "majority"): 100427,
}

for (target, policy), n_rows in expected.items():
    resolved = resolve_conflicts(train_df, target, policy)
    assert resolved.index.isin(train_df.index).all()
    # one target value per feature vector
    assert len(resolved.drop_duplicates(FEATURES)) == len(resolved.drop_duplicates(FEATURES + [target])), (target, policy)
    assert len(deduplicate(resolved)) == n_rows, (target, policy, len(deduplicate(resolved)))

In [8]:
assert len(resolve_conflicts(train_df)) == len(resolve_conflicts(train_df, "attack_cat", "majority"))

try:
    resolve_conflicts(train_df, policy="random")
except ValueError:
    pass
else:
    raise AssertionError("bad policy should raise ValueError")

### overlap_with_train

In [9]:
in_train = overlap_with_train(train_df, test_df)

assert in_train.dtype == bool
assert in_train.index.equals(test_df.index)
assert in_train.sum() == 8541, in_train.sum()

### distinct_rows

In [10]:
distinct = distinct_rows(test_df)

assert distinct.dtype == bool
assert distinct.index.equals(test_df.index)
assert distinct.sum() == 55945, distinct.sum()
assert distinct.sum() == len(test_df.drop_duplicates(FEATURES + ["attack_cat"]))
assert not test_df[distinct].duplicated(FEATURES + ["attack_cat"]).any()
# first copy kept, repeats dropped
row = test_df.iloc[[0]]
assert distinct_rows(pd.concat([row] * 3)).tolist() == [True, False, False]

# same features, different category: both count
other = "Worms" if row["attack_cat"].iloc[0] != "Worms" else "DoS"
assert distinct_rows(pd.concat([row, row.assign(attack_cat=other)])).tolist() == [True, True]

### Errors

In [11]:
try:
    load_unsw("validation")
except ValueError:
    pass
else:
    raise AssertionError("bad split should raise ValueError")

try:
    load_unsw("train", data_dir="C:/nu_exista")
except FileNotFoundError:
    pass
else:
    raise AssertionError("missing file should raise FileNotFoundError")

### train_val_split

Rows before the split: train cleaned for the target, one row per feature vector.

In [12]:
splits = {}
for target, n_rows in [("label", 100941), ("attack_cat", 100427)]:
    X_tr, X_val, y_tr, y_val = train_val_split(train_df, target)
    splits[target] = X_tr, X_val, y_tr, y_val

    assert len(X_tr) + len(X_val) == n_rows, (target, len(X_tr) + len(X_val))
    assert abs(len(X_val) / n_rows - 0.2) < 0.001
    assert list(X_tr.columns) == FEATURES
    assert X_tr.index.equals(y_tr.index) and X_val.index.equals(y_val.index)
    assert not X_tr.index.isin(X_val.index).any()
    assert not overlap_with_train(X_tr, X_val).any(), target

    share_tr = train_df.loc[X_tr.index, "attack_cat"].value_counts(normalize=True)
    share_val = train_df.loc[X_val.index, "attack_cat"].value_counts(normalize=True)
    assert (share_tr - share_val).abs().max() < 0.002, target

assert set(splits["label"][2].unique()) == {0, 1}
assert splits["attack_cat"][2].nunique() == 10

In [13]:
assert train_val_split(train_df, "label")[1].index.equals(splits["label"][1].index)
assert len(train_df) == 175341

## preprocessing.py

In [14]:
for target, (X_tr, X_val, y_tr, y_val) in splits.items():
    pre = build_preprocessor().fit(X_tr)
    names = list(pre.get_feature_names_out())
    Z_tr, Z_val, Z_test = (np.asarray(pre.transform(X)) for X in (X_tr, X_val, X_test))

    assert len(names) == Z_tr.shape[1] == Z_val.shape[1] == Z_test.shape[1] == 59, (target, Z_tr.shape)
    assert len(Z_test) == len(X_test)
    for Z in (Z_tr, Z_val, Z_test):
        assert np.isfinite(Z).all(), target

    num = Z_tr[:, :len(NUMERIC)]
    assert np.allclose(num.mean(axis=0), 0, atol=1e-6)
    assert np.allclose(num.std(axis=0), 1, atol=1e-6)

    for col in CATEGORICAL:
        idx = [i for i, n in enumerate(names) if n.startswith(f"cat__{col}_")]
        assert (Z_test[:, idx].sum(axis=1) == 1).all(), (target, col)

    unseen = X_test["state"].isin(["ACC", "CLO"]).to_numpy()
    assert unseen.sum() == 5
    assert (Z_test[unseen, names.index("cat__state_infrequent_sklearn")] == 1).all()

    assert np.array_equal(np.asarray(pre.transform(test_df)), Z_test)

### exclude

In [15]:
X_tr = splits["label"][0]
pre = build_preprocessor(exclude=TTL_COLUMNS).fit(X_tr)
names = list(pre.get_feature_names_out())

assert len(names) == 56, len(names)
assert not [n for n in names if any(col in n for col in TTL_COLUMNS)], names
assert {n.removeprefix("num__") for n in names if n.startswith("num__")} == set(NUMERIC) - set(TTL_COLUMNS)
assert np.asarray(pre.transform(X_test)).shape == (len(X_test), 56)

# a categorical column takes all its one-hot columns with it
assert len(build_preprocessor(exclude=["proto"]).fit(X_tr).get_feature_names_out()) == 59 - 6

try:
    build_preprocessor(exclude=["xyz"])
except ValueError:
    pass
else:
    raise AssertionError("unknown excluded column should raise ValueError")

## evaluation.py

Small hand-made example, metrics computed by hand: 4 normal, 4 attacks.

In [16]:
y = np.array([0, 0, 0, 0, 1, 1, 1, 1])
s = np.array([0.1, 0.2, 0.3, 0.8, 0.4, 0.7, 0.9, 0.95])

### evaluate_binary

In [17]:
m = evaluate_binary(y, s, threshold=0.5)

assert m["confusion_matrix"] == [[3, 1], [1, 3]], m["confusion_matrix"]
assert m["precision"] == m["recall"] == m["f1"] == 0.75
assert m["fpr"] == 0.25
assert m["roc_auc"] == 14 / 16
assert m["n"] == 8

m = evaluate_binary(y, s, threshold=1.0)
assert m["recall"] == 0 and m["precision"] == 0 and m["fpr"] == 0

assert evaluate_binary(np.ones(4, dtype=int), s[:4], 0.5)["roc_auc"] is None

### choose_threshold

In [18]:
assert choose_threshold(y, s, "fpr", max_fpr=0.0) == 0.9
assert choose_threshold(y, s, "fpr", max_fpr=0.25) == 0.4
assert choose_threshold(y, s, "f1") == 0.4

for budget in (0.0, 0.25, 0.5):
    t = choose_threshold(y, s, "fpr", max_fpr=budget)
    assert evaluate_binary(y, s, t)["fpr"] <= budget

constant = np.full(8, 0.57)
t = choose_threshold(y, constant, "fpr")
assert np.isfinite(t) and evaluate_binary(y, constant, t)["fpr"] == 0

try:
    choose_threshold(y, s, "accuracy")
except ValueError:
    pass
else:
    raise AssertionError("bad mode should raise ValueError")

#### normal_quantile

Only normal scores count. Normal scores in the toy example: 0.1, 0.2, 0.3, 0.8.

In [19]:
assert choose_threshold(y, s, "normal_quantile", max_fpr=0.25) == 0.8
assert choose_threshold(y, s, "normal_quantile", max_fpr=0.5) == 0.3
assert choose_threshold(y, s, "normal_quantile", max_fpr=1.0) == 0.1
t = choose_threshold(y, s, "normal_quantile", max_fpr=0.0)
assert t > 0.8 and evaluate_binary(y, s, t)["fpr"] == 0

# attack scores don't move it: shuffled, replaced, or no attacks at all
rng = np.random.default_rng(1)
for budget in (0.0, 0.25, 0.5):
    t = choose_threshold(y, s, "normal_quantile", max_fpr=budget)
    s_shuffled = s.copy()
    s_shuffled[y == 1] = rng.permutation(s[y == 1])
    s_other = s.copy()
    s_other[y == 1] = rng.uniform(size=4)
    assert choose_threshold(y, s_shuffled, "normal_quantile", max_fpr=budget) == t
    assert choose_threshold(y, s_other, "normal_quantile", max_fpr=budget) == t
    assert choose_threshold(y[y == 0], s[y == 0], "normal_quantile", max_fpr=budget) == t

# ties at the limit: a plain 80% quantile would be 0.5 and flag 4 of 10 normal flows
y_tie = np.array([0] * 10 + [1] * 3)
s_tie = np.array([0.1] * 6 + [0.5] * 3 + [0.9] + [0.2, 0.6, 0.95])
t = choose_threshold(y_tie, s_tie, "normal_quantile", max_fpr=0.2)
assert t == 0.9 and evaluate_binary(y_tie, s_tie, t)["fpr"] == 0.1
assert np.quantile(s_tie[:10], 0.8) == 0.5

# within budget on random data with many ties
for _ in range(200):
    yr = rng.integers(0, 2, 60)
    yr[0] = 0
    sr = rng.integers(0, 6, 60) / 5
    budget = rng.choice([0.0, 0.01, 0.1, 0.3])
    t = choose_threshold(yr, sr, "normal_quantile", max_fpr=budget)
    fpr = (sr[yr == 0] >= t).mean()
    assert fpr <= budget, (budget, fpr)
    # and it is the lowest such normal score: the next normal value below breaks the budget
    below = np.unique(sr[yr == 0])
    below = below[below < t]
    if len(below):
        assert (sr[yr == 0] >= below[-1]).mean() > budget

### evaluate_multiclass

In [20]:
mc = evaluate_multiclass(["a", "b", "b", "c"], ["a", "b", "c", "c"])

assert mc["labels"] == ["a", "b", "c"]
assert mc["accuracy"] == 0.75
assert mc["confusion_matrix"] == [[1, 0, 0], [0, 1, 1], [0, 0, 1]]
assert mc["per_class"]["b"] == {"precision": 1.0, "recall": 0.5, "f1": 2 / 3, "support": 2}
assert abs(mc["macro_f1"] - (1 + 2 / 3 + 2 / 3) / 3) < 1e-12

mc = evaluate_multiclass(["a", "a"], ["a", "a"], labels=["a", "b"])
assert mc["per_class"]["b"]["support"] == 0

# same numbers as sklearn on random labels (every class present and predicted)
rng = np.random.default_rng(0)
yt, yp = rng.choice(list("abcde"), 500), rng.choice(list("abcde"), 500)
mc_rand = evaluate_multiclass(yt, yp)
assert np.isclose(mc_rand["accuracy"], accuracy_score(yt, yp))
assert np.isclose(mc_rand["macro_f1"], f1_score(yt, yp, average="macro"))
assert np.isclose(mc_rand["weighted_f1"], f1_score(yt, yp, average="weighted"))

# a label outside `labels` would be dropped by confusion_matrix and inflate accuracy
for y_t, y_p in [(["a", "a", "b"], ["a", "x", "b"]), (["a", "x", "b"], ["a", "a", "b"])]:
    try:
        evaluate_multiclass(y_t, y_p, labels=["a", "b"])
    except ValueError:
        pass
    else:
        raise AssertionError("unknown label should raise ValueError")

### detection_by_category

Same toy example, with an attack category for each row.

In [21]:
cats = np.array(["Normal"] * 4 + ["DoS", "DoS", "Worms", "Worms"])
m = evaluate_binary(y, s, threshold=0.5, categories=cats)

assert m["by_category"] == {
    "DoS": {"n": 2, "flagged": 0.5},
    "Normal": {"n": 4, "flagged": 0.25},
    "Worms": {"n": 2, "flagged": 1.0},
}
assert m["by_category"]["Normal"]["flagged"] == m["fpr"]
attacks = [v for k, v in m["by_category"].items() if k != "Normal"]
assert sum(v["n"] * v["flagged"] for v in attacks) / sum(v["n"] for v in attacks) == m["recall"]
assert "by_category" not in evaluate_binary(y, s, 0.5)
assert detection_by_category(cats, s >= 0.5) == m["by_category"]

# macro_detection: mean over attack categories, Normal left out
assert m["macro_detection"] == (0.5 + 1.0) / 2
cats2 = np.array(["Normal"] * 4 + ["DoS", "Worms", "Worms", "Worms"])
m2 = evaluate_binary(y, s, threshold=0.5, categories=cats2)
assert m2["by_category"]["DoS"]["flagged"] == 0.0 and m2["by_category"]["Worms"]["flagged"] == 1.0
assert m2["macro_detection"] == 0.5
assert m2["recall"] == 0.75
assert "macro_detection" not in evaluate_binary(y, s, 0.5)
assert evaluate_binary(np.zeros(4, dtype=int), s[:4], 0.5, categories=cats[:4])["macro_detection"] is None

### fpr_by_group

Normal flows only, split in groups; group c exists only in test.

In [22]:
g_val, f_val = ["a", "a", "b", "b"], [False, True, False, False]
g_test, f_test = ["a", "b", "b", "c", "c", "c"], [True, False, True, False, False, True]
t = fpr_by_group(g_val, f_val, g_test, f_test)

assert list(t.index) == ["a", "b", "c"]
assert t[("val", "flows")].tolist() == [2, 2, 0] and t[("test", "flows")].tolist() == [1, 2, 3]
assert t[("val", "share")].tolist() == [0.5, 0.5, 0.0]
assert t[("val", "fpr")].iloc[:2].tolist() == [0.5, 0.0] and np.isnan(t.loc["c", ("val", "fpr")])
assert t[("test", "fpr")].tolist() == [1.0, 0.5, 1 / 3]
# contributions add up to the overall test FPR
assert np.isclose(t[("test", "fpr contribution")].sum(), np.mean(f_test))
assert np.isclose(t[("test", "share")].sum(), 1)

### evaluate_test

In [23]:
unseen = np.array([True, False, True, False, True, False, True, False])
dist = np.array([True, True, True, False, True, True, False, False])
r = evaluate_test(evaluate_binary, pd.Series(y), s, unseen, dist, threshold=0.5)

assert set(r) == {"test", "test_unseen", "test_distinct"}
assert r["test"] == evaluate_binary(y, s, 0.5)
assert r["test_unseen"] == evaluate_binary(y[unseen], s[unseen], 0.5)
assert r["test_distinct"] == evaluate_binary(y[dist], s[dist], 0.5)
assert r["test_distinct"]["n"] == 5

r = evaluate_test(evaluate_binary, y, s, unseen, dist, categories=cats, threshold=0.5)
assert r["test_unseen"]["by_category"] == evaluate_binary(y[unseen], s[unseen], 0.5, categories=cats[unseen])["by_category"]
assert sum(v["n"] for v in r["test_unseen"]["by_category"].values()) == unseen.sum()
assert r["test_distinct"]["by_category"] == evaluate_binary(y[dist], s[dist], 0.5, categories=cats[dist])["by_category"]
assert r["test_distinct"]["macro_detection"] == evaluate_binary(y[dist], s[dist], 0.5, categories=cats[dist])["macro_detection"]

r = evaluate_test(evaluate_multiclass, ["a", "b", "b", "c"], ["a", "b", "c", "c"], [True, True, False, False], [True, False, True, True])
assert r["test_unseen"]["n"] == 2
assert r["test_distinct"]["n"] == 3 and r["test_distinct"]["accuracy"] == 2 / 3

### save_result

Writes to a temporary file, `reports/results.json` is not touched.

In [24]:
with tempfile.TemporaryDirectory() as tmp:
    out = Path(tmp) / "results.json"
    metrics = {"val": evaluate_binary(y, s, 0.5)}

    save_result("A", "binary", metrics, {"x": 1}, path=out)
    save_result("B", "binary", metrics, path=out)
    save_result("A", "binary", metrics, {"x": 2}, path=out)
    save_result("A", "multiclass", {"val": mc}, path=out)

    saved = json.loads(out.read_text(encoding="utf-8"))
    assert [(r["model"], r["task"]) for r in saved] == [("B", "binary"), ("A", "binary"), ("A", "multiclass")]
    assert saved[1]["params"] == {"x": 2}
    assert saved[0]["params"] == {}
    assert saved[1]["val"] == metrics["val"]

### results_table

In [25]:
with tempfile.TemporaryDirectory() as tmp:
    out = Path(tmp) / "results.json"
    b = evaluate_test(evaluate_binary, y, s, unseen, dist, categories=cats, threshold=0.5)
    save_result("A", "binary", {"val": evaluate_binary(y, s, 0.3), **b}, path=out)
    save_result("B", "binary", {"val": evaluate_binary(y, 1 - s, 0.5), **b}, path=out)
    save_result("A", "multiclass", {"val": mc, "test": mc}, path=out)

    t = results_table("binary", path=out)
    assert list(t.index) == ["A", "B"], t.index
    assert list(t.columns) == ["recall", "macro_detection", "fpr", "precision", "f1", "roc_auc", "pr_auc"], t.columns
    assert t.loc["A", "recall"] == b["test"]["recall"]
    assert t.loc["A", "macro_detection"] == b["test"]["macro_detection"]

    t = results_table("binary", "val", ["recall", "fpr"], path=out)
    assert t.loc["A", "recall"] == evaluate_binary(y, s, 0.3)["recall"]
    assert t.loc["B", "fpr"] == evaluate_binary(y, 1 - s, 0.5)["fpr"]

    t = results_table("binary", "test_unseen", path=out)
    assert t.loc["B", "f1"] == b["test_unseen"]["f1"]

    t = results_table("binary", "test_distinct", ["recall", "macro_detection"], path=out)
    assert t.loc["B", "recall"] == b["test_distinct"]["recall"]

    # val was saved without categories: no macro_detection, shown as NaN
    assert results_table("binary", "val", path=out)["macro_detection"].isna().all()

    t = results_table("multiclass", path=out)
    assert list(t.index) == ["A"] and list(t.columns) == ["accuracy", "macro_f1", "weighted_f1"]
    assert t.loc["A", "macro_f1"] == mc["macro_f1"]

### Plots

In [26]:
with tempfile.TemporaryDirectory() as tmp:
    fig = plot_roc_pr(y, s, "toy", path=Path(tmp) / "roc.png")
    assert (Path(tmp) / "roc.png").exists()
    plt.close(fig)

    fig = plot_confusion(mc["confusion_matrix"], mc["labels"], path=Path(tmp) / "cm.png")
    assert (Path(tmp) / "cm.png").exists()
    plt.close(fig)

## models/autoencoder.py

Synthetic data: "normal" rows lie on a 3-dimensional subspace of 12 columns, "attacks" are off it.

In [27]:
rng = np.random.default_rng(0)
basis = rng.normal(size=(3, 12))
normal = (rng.normal(size=(2000, 3)) @ basis).astype(np.float32)
normal_tr, normal_val = normal[:1600], normal[1600:]
attacks = rng.normal(scale=2.0, size=(200, 12)).astype(np.float32)

### Autoencoder

In [28]:
ae = Autoencoder(12, bottleneck=4)
assert ae(torch.zeros(5, 12)).shape == (5, 12)
assert ae.encoder(torch.zeros(5, 12)).shape == (5, 4)
assert [m.out_features for m in ae.encoder if isinstance(m, torch.nn.Linear)] == [32, 16, 4]
assert [m.out_features for m in ae.decoder if isinstance(m, torch.nn.Linear)] == [16, 32, 12]
# linear output: reconstructions can be negative
assert not isinstance(ae.decoder[-1], torch.nn.ReLU) and not isinstance(ae.encoder[-1], torch.nn.ReLU)

### reconstruction_error

In [29]:
err = reconstruction_error(ae, normal_val)

# one score per row, not per batch
assert err.shape == (len(normal_val),), err.shape
assert np.allclose(err, reconstruction_error(ae, normal_val, batch_size=7))
with torch.no_grad():
    x = torch.as_tensor(normal_val[:5])
    by_hand = ((ae(x) - x) ** 2).mean(dim=1).numpy()
assert np.allclose(err[:5], by_hand)
assert not ae.training

### train_autoencoder

In [30]:
model, history = train_autoencoder(normal_tr, normal_val, bottleneck=3, epochs=60, batch_size=64, patience=5)

val_losses = [h["val_loss"] for h in history]
assert not model.training
assert history[0]["epoch"] == 1 and len(history) <= 60
assert val_losses[-1] < val_losses[0] / 2, val_losses
assert np.isclose(reconstruction_error(model, normal_val).mean(), min(val_losses), rtol=1e-5)

# with a high learning rate the validation loss bounces and training stops early
noisy, noisy_history = train_autoencoder(normal_tr, normal_val, bottleneck=3, epochs=60, batch_size=64, patience=5, lr=0.05)
noisy_losses = [h["val_loss"] for h in noisy_history]
assert len(noisy_history) < 60, len(noisy_history)
assert np.argmin(noisy_losses) == len(noisy_history) - 1 - 5  # stopped `patience` epochs after the best
# the weights kept are the best epoch's, not the last
assert noisy_losses[-1] > min(noisy_losses)
assert np.isclose(reconstruction_error(noisy, normal_val).mean(), min(noisy_losses), rtol=1e-5)

# attacks are off the subspace: much larger error
assert np.median(reconstruction_error(model, attacks)) > 10 * np.median(reconstruction_error(model, normal_val))

# same seed, same model
model2, history2 = train_autoencoder(normal_tr, normal_val, bottleneck=3, epochs=60, batch_size=64, patience=5)
assert history2 == history
assert np.array_equal(reconstruction_error(model2, attacks), reconstruction_error(model, attacks))

### save_autoencoder / load_autoencoder

In [31]:
with tempfile.TemporaryDirectory() as tmp:
    save_autoencoder(model, pre, 0.123, directory=tmp, name="ae")
    assert {p.name for p in Path(tmp).iterdir()} == {"ae.pt", "ae_preprocessor.joblib", "ae.json"}

    loaded, loaded_pre, threshold = load_autoencoder(tmp, name="ae")
    assert threshold == 0.123
    assert (loaded.n_features, loaded.bottleneck, loaded.hidden) == (12, 3, (32, 16))
    assert not loaded.training
    assert np.array_equal(reconstruction_error(loaded, attacks), reconstruction_error(model, attacks))
    assert np.array_equal(np.asarray(loaded_pre.transform(X_test[:100])), np.asarray(pre.transform(X_test[:100])))

In [32]:
print("all checks passed")

all checks passed
